# SGD Scratch

In [3]:
import numpy as np

def norm_pdf(X, mean, log_var):
    D = X.shape[1]
    var = np.exp(log_var)
    diff = X - mean
    exponent = -0.5 * np.sum((diff ** 2) / var, axis=1)
    log_norm = -0.5 * (D * np.log(2 * np.pi) + np.sum(log_var))
    return np.exp(exponent + log_norm) 

def responsibilities(X, means, log_vars, weights):
    K = len(means)
    N = X.shape[0]
    probs = np.zeros((N, K))
    for k in range(K):
        probs[:, k] = weights[k] * norm_pdf(X, means[k], log_vars[k])

    denom = np.sum(probs, axis=1, keepdims=True) + 1e-10
    return probs / denom 

def mean_gradient(X, resp, k, mean, log_var):
    var = np.exp(log_var)
    diff = X - mean
    grad = np.sum(resp[:, k][:, None] * (diff / var), axis=0)
    return grad

def logvar_gradient(X, resp, k, mean, log_var):
    var = np.exp(log_var)
    diff = X - mean
    grad = 0.5 * np.sum(
        resp[:, k][:, None] * ((diff ** 2) / var - 1),
        axis=0
    )
    return grad

def mw_gradient(resp, weights, k):
    grad = np.sum(resp[:, k]) / (weights[k] + 1e-10)
    return grad

def sgd_step(X, means, log_vars, log_weights, lr=0.01, batch_size=64):

    N = X.shape[0]
    batch_idx = np.random.choice(N, batch_size, replace=False)
    Xb = X[batch_idx]

    exp_w = np.exp(log_weights - np.max(log_weights))
    weights = exp_w / np.sum(exp_w)

    resp = responsibilities(Xb, means, log_vars, weights)

    for k in range(len(means)):
        g_mu = mean_gradient(Xb, resp, k, means[k], log_vars[k])
        g_logvar = logvar_gradient(Xb, resp, k, means[k], log_vars[k])
        g_w = mw_gradient(resp, weights, k)

        means[k] += lr * g_mu
        log_vars[k] += lr * g_logvar
        log_weights[k] += lr * g_w

    return means, log_vars, log_weights

def nll(X, means, log_vars, weights):
    # negative log likelihood
    N = X.shape[0]
    K = len(means)
    probs = np.zeros((N, K))
    for k in range(K):
        probs[:, k] = weights[k] * norm_pdf(X, means[k], log_vars[k])
    return -np.sum(np.log(np.sum(probs, axis=1) + 1e-10))


In [9]:
import numpy as np

feature_dims=2
components=3
num_samples=1000
means = np.random.uniform(-5,5,size=(components,feature_dims))
covariances = [
    np.diag(np.random.uniform(0.5,1.5,size=feature_dims)) for _ in range(components)
]
np.random.seed(0)

X=[]
y=[]

for i in range(components):
    samples = np.random.multivariate_normal(means[i], covariances[i], size=num_samples//components)
    X.append(samples)
    y.append(np.full((num_samples//components,), i))

X = np.vstack(X)
y = np.hstack(y)

K, D = 3, 2
np.random.seed(0)
means = np.random.randn(K, D)
log_vars = np.zeros((K, D))   # variance=1
weights = np.ones(K) / K

lr = 1e-2
epochs = 200

for epoch in range(epochs):
    resp = responsibilities(X, means, log_vars, weights)

    for k in range(K):
        g_mean = mean_gradient(X, resp, k, means[k], log_vars[k])
        g_logvar = logvar_gradient(X, resp, k, means[k], log_vars[k])
        g_weight = mw_gradient(resp, weights, k)

        means[k] += lr * g_mean
        log_vars[k] += lr * g_logvar
        weights[k] += lr * g_weight

    # normalize weights
    weights = np.maximum(weights, 1e-8)
    weights /= np.sum(weights)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, NLL = {nll(X, means, log_vars, weights):.2f}")

print("\nLearned means:\n", means)
print("\nLearned variances:\n", np.exp(log_vars))
print("\nLearned weights:\n", weights)


Epoch 20, NLL = 4966.76
Epoch 40, NLL = 3862.90
Epoch 60, NLL = 5719.96
Epoch 80, NLL = 5216.28
Epoch 100, NLL = 3994.65
Epoch 120, NLL = 4349.24
Epoch 140, NLL = 4202.54
Epoch 160, NLL = 3902.19
Epoch 180, NLL = 4250.16
Epoch 200, NLL = 4126.20

Learned means:
 [[ 2.45619039 -0.21388268]
 [-0.55369058  3.33278925]
 [ 2.77872243  0.20197118]]

Learned variances:
 [[2.96077768 0.49782537]
 [2.1166103  1.50825564]
 [1.77950503 0.91659985]]

Learned weights:
 [0.19659127 0.29858258 0.50482615]


In [ ]:
def norm_pdf(X, mean, cov):
    d = X.shape[1]
    cov_inv = np.linalg.inv(cov)
    diff = X - mean
    exponent = -0.5 * np.sum(diff @ cov_inv * diff, axis=1)
    denom = np.sqrt((2 * np.pi) ** d * np.linalg.det(cov))
    return np.exp(exponent) / denom

def responsibilities(X,means,covariances,weights):
    output = np.zeros((X.shape[0], len(means)))
    denom = np.dot(weights, [norm_pdf(X, means[k], covariances[k]) for k in range(len(means))])

    for k in range(len(means)):
        numer = weights[k] * norm_pdf(X, means[k], covariances[k])
        output[:,k] = numer / denom
    return output

def mean_gradient(X, resp, k, mean, cov):
    conv_inv = np.linalg.inv(cov)
    diff = X - mean
    grad = np.sum(resp[:,k][:,np.newaxis] * (diff @ conv_inv), axis=0)
    return grad

def logvar_gradient(X, resp, k, mean, log_var):

    var = np.exp(log_var)       
    diff = X - mean            
    grad = 0.5 * np.sum(resp[:, k][:, None] * ((diff ** 2) / var - 1),axis=0)
    return grad

def mw_gradient(resp, weights, k):
    return np.sum(resp[:,k] / weights[k])